[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/two_spins/two_spins.ipynb)

# Two Spins

This example computes exchange entanglement, Bell violations, measurement steering, Zeeman level crossings, and singlet-triplet precession for two coupled spins. Starting with one spin up and one spin down, the exchange interaction swaps the two spins while entangling them fully halfway through. At maximal entanglement neither spin shows a turning plane of its own, yet measurements in matching planes always disagree and steer the second spin across its entire sphere. A uniform magnetic field tilts the spin-aligned triplet below the entangled singlet, whereas a differential field between the two sites drives coherent oscillations inside the zero-spin subspace.

Each spin is modelled in its own copy of spacetime with observer time directions `t` and `T`, so every element belonging to one spin commutes with every element of the other. Single-spin states are spacelike even multivectors of their own copy, and the pair state lies in the ideal of `correlator = 0.5 * (1 - xy * XY)`, where right multiplication by either spin's xy plane gives the shared `reference_spin = correlator * xy`. Spins, magnetic fields, and measurement settings are all spatial turning planes without pseudoscalars; full-algebra duals span both copies at once, while an even spatial reflection `x * X` mirrors both copies consistently. Projections of `2 * (state >> reference_spin)` give each spin's turning plane, and `(density * SecondPlanes).cast(FirstPlanes)` forms the joint spin map whose singular values determine concurrence and the largest Bell combination, while its adjoint steers the second spin upon measurement. A conventional vector-space twin in [two_spins_space.ipynb](two_spins_space.ipynb) runs the same experiments with axial Bloch vectors and demonstrates how spatial pseudoscalars invert under reflection.

In bra-ket and Pauli-matrix notation, the pair state is written as a four-component ket $|\psi\rangle \in \mathbb{C}^2 \otimes \mathbb{C}^2$ acted upon by Kronecker products $\sigma_i \otimes \sigma_j$ with an explicit imaginary unit $i$, single-spin planes require partial traces $\rho_1 = \operatorname{tr}_2 |\psi\rangle\langle\psi|$ followed by $\mathbf a = \operatorname{tr}(\rho_1 \boldsymbol\sigma)$, and joint spin expectations assemble a separate $3 \times 3$ component matrix $T_{ij} = \langle\psi| \sigma_i \otimes \sigma_j |\psi\rangle$ alongside $4 \times 4$ matrix exponentials $e^{-iJt\,\boldsymbol\sigma_1\cdot\boldsymbol\sigma_2}$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
import numpy as np

from numga import Algebra, NumpyContext, stack
from examples.util.animation import save_figure, show_animation
from examples.quantum.two_spins import render

np.set_printoptions(precision=4, suppress=True)

# Two copies of spacetime, one per spin, and the observer's spatial subalgebra in each copy.
ga = Algebra("t-x+y+z+T-X+Y+Z+")
first_space = Algebra("x+y+z+")
second_space = Algebra("X+Y+Z+")
context = NumpyContext(ga)
mv = context.multivector
Scalar = ga.gatype.scalar()                                                     # Scalar+
Bivector = ga.gatype.bivector()                                                 # Bivector+
FirstPlanes = ga.gatype.bivector().derive.within(first_space)                   # Bivector+
SecondPlanes = ga.gatype.bivector().derive.within(second_space)                 # Bivector+
FirstSpinor = ga.gatype.even().derive.within(first_space)                       # Even+
SecondSpinor = ga.gatype.even().derive.within(second_space)                     # Even+
Spinor = ga.gatype(FirstSpinor * SecondSpinor)                                  # Even+
JointMap = ga.gatype((FirstPlanes, SecondPlanes))                               # FirstPlanes+ <- SecondPlanes+

# Correlator idempotent and shared reference spin on the right.
correlator = 0.5 * (1 - mv.xy * mv.XY)                                          # [] Spinor+
reference_spin = correlator * mv.xy                                             # [] Spinor+
# Exchange operator and its orthogonal singlet and triplet projectors.
coupling = mv.yz * mv.YZ + mv.zx * mv.ZX + mv.xy * mv.XY                        # [] Spinor+
singlet_part = 0.25 * (1 + coupling)                                            # [] Spinor+
triplet_part = 0.25 * (3 - coupling)                                            # [] Spinor+
# Uniform and differential magnetic field planes in the two xy planes.
field_plane = mv.xy + mv.XY                                                     # [] Bivector+
difference_plane = 0.5 * (mv.xy - mv.XY)                                        # [] Bivector+

## Two spins in two copies of spacetime

Each single-spin state is an even spacelike multivector in its own copy, commuting with both observer time directions `t` and `T`. Spin up turns in `xy` with state 1, while a half turn `-ZX` flips the second spin's turning plane to `-XY` so `-ZX * correlator` prepares the first spin up and the second spin down. Within the ideal of `correlator`, right multiplication by `xy` or `XY` coincides with `reference_spin`, which squares to `-correlator` and generates shared quantum phase rotations from the right. Every normalized pair state has unit scalar part in `2 * state.symmetric_reverse_product()`.

In bra-ket notation, the pair state reads as $|\psi\rangle \in \mathbb{C}^2 \otimes \mathbb{C}^2$, spin up and spin down read as $|0\rangle$ and $|1\rangle$, and right multiplication by `reference_spin` corresponds to multiplication by $i$.

In [ ]:
# First spin up in xy and second spin down via a half turn in ZX.
up_down = -mv.ZX * correlator                                                   # [] Spinor+
state_norm = 2 * up_down.scalar_norm_squared()                                  # [] Scalar+
print("norm of initial up-down state:", state_norm.to_array())

## Spin planes and joint expectations

Sandwiching `reference_spin` as `2 * (state >> reference_spin)` produces the pair's bivector spin, whose projections onto `FirstPlanes` and `SecondPlanes` give the plane each spin turns in on its own. Each plane has unit norm when its spin is unentangled and shrinks as entanglement grows. Multiplying the state density `2 * state.symmetric_reverse_product()` by `SecondPlanes` and projecting onto `FirstPlanes` extracts the joint spin map `joint_map(state)`, for which `first_plane.dot(joint(second_plane))` equals the expected product of measurement outcomes in unit planes `first_plane` and `second_plane`. From the two largest singular values of `joint` follows the largest Bell combination achievable over all measurement settings, which equals 2 on any unentangled product state.

In bra-ket notation, the single-spin plane reads as the Bloch vector $\operatorname{tr}(\rho_1 \boldsymbol\sigma)$ of the reduced density matrix $\rho_1 = \operatorname{tr}_2 |\psi\rangle\langle\psi|$, the joint spin map reads as the spin-pair expectation matrix $T_{ij} = \langle\psi| \sigma_i \otimes \sigma_j |\psi\rangle$, and the largest Bell combination reads as $2\sqrt{t_1^2 + t_2^2}$ in terms of the singular values $t_i$ of $T$.

In [ ]:
def spin_planes(state: Spinor) -> tuple[FirstPlanes, SecondPlanes]:
    """The pair spin `2 * (state >> reference_spin)` projects onto each spin's spatial planes (`[...] FirstPlanes+, SecondPlanes+`)."""
    pair_spin = 2 * (state >> reference_spin)                                   # [...] Bivector+
    return pair_spin.cast(FirstPlanes), pair_spin.cast(SecondPlanes)


def joint_map(state: Spinor) -> JointMap:
    """The grade-(2, 2) block of the state density defines the joint spin map (`[...] FirstPlanes+ <- SecondPlanes+`) between the two spins' planes."""
    density = 2 * state.symmetric_reverse_product()                             # [...] Spinor+
    return (density * SecondPlanes).cast(FirstPlanes)                           # [...] FirstPlanes+ <- SecondPlanes+


start_first, start_second = spin_planes(up_down)                                # [] FirstPlanes+, SecondPlanes+
start_joint = joint_map(up_down)                                                # [] FirstPlanes+ <- SecondPlanes+
plane_reading = render.Reading(
    stack((mv.yz, mv.zx, mv.xy)),
    stack((mv.YZ, mv.ZX, mv.XY)),
    "size of each spin's plane",
)
save_figure(render.draw_state(start_first, start_second, plane_reading), "two_spins_state");

Each sphere plots the three plane components along `yz`, `zx`, and `xy` of one spin's space. At the initial product state, the red arrow places the first spin at unit norm along `+xy` and the blue arrow places the second spin at unit norm along `-XY`.

In [ ]:
def bell_maximum(joint: JointMap) -> Scalar:
    """The two largest singular values of `joint` set the largest Bell combination (`[...] Scalar+`) over all measurement planes."""
    values = joint.svdvals()                                                    # [..., 3] Scalar+
    return 2 * (values[..., 0] ** 2 + values[..., 1] ** 2).square_root()        # [...] Scalar+


start_bell = bell_maximum(start_joint)                                          # [] Scalar+
print("largest Bell combination on initial up-down state:", start_bell.to_array())

## Exchange entanglement

Because `coupling * coupling == 3 + 2 * coupling`, the exchange operator has eigenvalues 3 on the singlet subspace and -1 on the triplet subspace, with orthogonal idempotent projectors `singlet_part = (1 + coupling) / 4` and `triplet_part = (3 - coupling) / 4`. Under exchange for angle `angle`, the singlet part turns on the right by `3 * angle` in `xy` while the triplet part turns backward by `angle`. Because each term in `coupling` squares to +1 (`(yz * YZ) * (yz * YZ) == 1`), a commutator with `coupling` alone stretches density elements without turning them, whereas right multiplication by `reference_spin` generates unitary phase rotation between the singlet and triplet components of `up_down`.

In bra-ket notation, exchange evolution reads as $e^{-i\theta\,\boldsymbol\sigma_1\cdot\boldsymbol\sigma_2}$, splitting into the singlet projector $\tfrac14\left(1 - \boldsymbol\sigma_1\cdot\boldsymbol\sigma_2\right)$ and the triplet projector $\tfrac14\left(3 + \boldsymbol\sigma_1\cdot\boldsymbol\sigma_2\right)$.

In [ ]:
def exchange_state(state: Spinor, angle: np.ndarray) -> Spinor:
    """Exchange evolution over `angle` turns the singlet part forward by `3 * angle` and the triplet part backward by `angle` (`[...] Spinor+`)."""
    singlet_turn = (mv.xy * (3 * angle)).exp()                                  # [...] Rotor+
    triplet_turn = (mv.xy * -angle).exp()                                       # [...] Rotor+
    return singlet_part * state * singlet_turn + triplet_part * state * triplet_turn   # [...] Spinor+


n_angles = 181
midpoint = n_angles // 2
angles = np.linspace(0.0, np.pi / 4, n_angles)
states = exchange_state(up_down, angles)                                        # [n_angles] Spinor+
first_planes, second_planes = spin_planes(states)                               # [n_angles] FirstPlanes+, SecondPlanes+
joints = joint_map(states)                                                      # [n_angles] FirstPlanes+ <- SecondPlanes+
bells = bell_maximum(joints)                                                    # [n_angles] Scalar+
save_figure(render.draw_exchange(angles, bells, first_planes, plane_reading), "two_spins_exchange");

As the exchange angle runs from `0` to `pi / 4`, the red curve on the left shows each individual spin's plane shrinking from unit size to zero at the midpoint before returning to unit size once the two spins have swapped. On the right, the purple curve rises above the dashed local-realist bound at 2 and reaches the dotted Tsirelson ceiling `2 * sqrt(2)` halfway through the swap.

## Singlet state and the Bell bound

Normalizing `singlet_part * up_down` isolates the singlet state, whose joint spin map sends every spatial plane of the second spin to the reversed plane of the first. For two unit planes on each spin, the Bell combination is the expectation of `bell_element`, built from products `first_plane * second_plane` that represent minus the product of the two spins' outcomes. Squaring `bell_element` yields `4 - 4 * first_choice.commutator(first_alt) * second_choice.commutator(second_alt)`. Because the commutator of two unit planes has norm equal to the sine of the angle between them, the square never exceeds 8 and no quantum state exceeds `2 * sqrt(2)`. Orthogonal planes on the first spin paired with diagonal planes on the second saturate that bound on the singlet.

In bra-ket notation, the Bell element reads as the CHSH operator ([Clauser, Horne, Shimony, and Holt, 1969](https://doi.org/10.1103/PhysRevLett.23.880)) $\mathcal{B} = A \otimes (B + B') + A' \otimes (B - B')$ with operator square $\mathcal{B}^2 = 4\,I - [A, A'] \otimes [B, B']$, bounding $\langle\mathcal{B}\rangle$ by $2\sqrt{2}$.

In [ ]:
def bell_element(
    first_plane: FirstPlanes,
    first_alt: FirstPlanes,
    second_plane: SecondPlanes,
    second_alt: SecondPlanes,
) -> Spinor:
    """Two measurement planes on each spin combine into the left-acting Bell multivector (`[...] Spinor+`)."""
    return -(first_plane * (second_plane + second_alt) + first_alt * (second_plane - second_alt))   # [...] Spinor+


def expectation(element: Spinor, state: Spinor) -> Scalar:
    """Sandwiching a left-acting multivector `element` from the reverse side gives its scalar expectation (`[...] Scalar+`) in `state`."""
    return 2 * (state << element).select[0]                                     # [...] Scalar+


singlet = np.sqrt(2) * singlet_part * up_down                                   # [] Spinor+
singlet_joint = joint_map(singlet)                                              # [] FirstPlanes+ <- SecondPlanes+
rng = np.random.default_rng(0)
first_choice, first_alt = mv(FirstPlanes, rng.normal(size=(2, 3))).normalized()   # [] FirstPlanes+, FirstPlanes+
second_choice, second_alt = mv(SecondPlanes, rng.normal(size=(2, 3))).normalized() # [] SecondPlanes+, SecondPlanes+
sample_element = bell_element(first_choice, first_alt, second_choice, second_alt) # [] Spinor+
element_square = sample_element * sample_element                                # [] Spinor+
commutator_bound = 4 - 4 * first_choice.commutator(first_alt) * second_choice.commutator(second_alt)   # [] Spinor+
half = 1 / np.sqrt(2)
singlet_bell = expectation(bell_element(mv.yz, mv.zx, -half * (mv.YZ + mv.ZX), -half * (mv.YZ - mv.ZX)), singlet)   # [] Scalar+
print("Bell combination on the singlet:", singlet_bell.to_array(), "  Tsirelson bound:", 2 * np.sqrt(2))

## Steering by measurement

Measuring outcome +1 for the first spin in a unit plane `plane` projects the pair onto `0.5 * (state - plane * state * reference_spin)` with probability `0.5 * (1 + plane.dot(first_plane))`. Renormalizing the projected state leaves the second spin in `(second_plane + joint.adjoint()(plane)) / (1 + plane.dot(first_plane))`, showing that the adjoint of the joint spin map steers the unmeasured spin. Because outcome -1 in `plane` coincides with outcome +1 in `-plane`, sampling unit planes over the full sphere covers both outcomes.

In bra-ket notation, the projective measurement applies $\tfrac12(I + \mathbf n\cdot\boldsymbol\sigma)\otimes I$, mapping the second spin's Bloch vector to $(\mathbf b + T^{\mathsf T}\mathbf n)/(1 + \mathbf n\cdot\mathbf a)$.

In [ ]:
def steered_spin(state: Spinor, plane: FirstPlanes) -> tuple[Scalar, SecondPlanes]:
    """Projecting the first spin onto outcome +1 in unit `plane` gives the outcome probability (`[...] Scalar+`) and the steered second spin plane (`[...] SecondPlanes+`)."""
    # Left multiplication by `-plane` and right multiplication by `reference_spin` gives the +1/-1 observable in `plane`.
    projected_state = 0.5 * (state - plane * state * reference_spin)            # [...] Spinor+
    probability = 2 * projected_state.scalar_norm_squared()                     # [...] Scalar+
    _, second_plane = spin_planes(projected_state / probability.square_root())  # [...] SecondPlanes+
    return probability, second_plane


frame_duration_ms = 40
n_around, n_down = 24, 12
polar = (np.arange(n_down) + 0.5) / n_down * np.pi
azimuth = np.arange(n_around) / n_around * 2 * np.pi
polar_rotors = (mv.zx * (-polar / 2)).exp()                                     # [n_down] Rotor+
azimuth_rotors = (mv.xy * (-azimuth / 2)).exp()                                 # [n_around] Rotor+
polar_planes = polar_rotors >> mv.xy                                            # [n_down] FirstPlanes+
planes = azimuth_rotors >> polar_planes[:, None]                                # [n_down, n_around] FirstPlanes+
probability, steered_planes = steered_spin(states[:, None, None], planes)       # [n_angles, n_down, n_around] Scalar+, SecondPlanes+
adjoint_planes = (second_planes[:, None, None] + joints[:, None, None].adjoint()(planes)) / (
    1 + planes.dot(first_planes[:, None, None])
)                                                                               # [n_angles, n_down, n_around] SecondPlanes+
show_animation(render.animate_steering(planes, probability, steered_planes, plane_reading), "two_spins_steering", frame_duration_ms)

RGB-coloured dots on the left sphere mark the sampled measurement planes of the first spin, while matching dots on the right sphere show where outcome +1 steers the second spin's plane, with area proportional to probability. At the start every measurement leaves the second spin pinned at `-XY`, halfway through the swap the steered states cover the entire unit sphere, and at the end they collapse to `+XY`.

## Entanglement in the singular values

Turning each spin locally by rotors `first_turn` and `second_turn` from the left transforms the joint spin map by sandwiching from both sides, `joint_map(first_turn * second_turn * state) == first_turn >> joints(second_turn << SecondPlanes)`. Because orthogonal turns on either side preserve singular values, the singular values of `joints` are local invariants of entanglement: 1 together with the concurrence `concurrence` repeated twice. Each spin's plane has norm `(1 - concurrence**2).square_root()`, so single-spin visibility trades directly against pair entanglement, while the largest Bell combination equals `2 * (1 + concurrence**2).square_root()`.

In bra-ket notation, local unitaries $U_1 \otimes U_2$ act on the expectation matrix via $\mathrm{SO}(3)$ rotations $T \mapsto R_1 T R_2^{\mathsf T}$, and pure-state concurrence is written via spin-flipped overlap $C = |\langle\psi|\,\sigma_y\otimes\sigma_y\,|\psi^*\rangle|$ ([Wootters, 1998](https://arxiv.org/abs/quant-ph/9709029)).

In [ ]:
# Independent random local spatial rotors acting on each spin copy.
first_turn = mv(FirstPlanes, rng.normal(size=3)).exp()                          # [] Rotor+
second_turn = mv(SecondPlanes, rng.normal(size=3)).exp()                        # [] Rotor+
turned_joints = joint_map(first_turn * second_turn * states)                    # [n_angles] FirstPlanes+ <- SecondPlanes+
sandwich_joints = first_turn >> joints(second_turn << SecondPlanes)             # [n_angles] FirstPlanes+ <- SecondPlanes+
singular_values = joints.svdvals()                                              # [n_angles, 3] Scalar+
turned_singular_values = turned_joints.svdvals()                                # [n_angles, 3] Scalar+
concurrence = singular_values[..., 1]                                           # [n_angles] Scalar+
save_figure(render.draw_entanglement(angles, singular_values, first_planes, plane_reading), "two_spins_entanglement");

Across the exchange sweep, the solid purple curve shows the leading singular value pinned at 1, the dashed purple curve traces the twofold-degenerate concurrence rising to 1 at the midpoint, and the solid red curve shows the complementary single-spin plane size dropping to 0.

## Energy levels in a field

A uniform magnetic field in the spatial `xy` planes is represented by `field_plane = xy + XY` and lowers the energy in proportion to `field_plane.dot(first_plane + second_plane)`. Together with the exchange interaction shifted to `-exchange_rate` on the singlet and `+exchange_rate` on the triplets, the singlet and the three definite-spin triplets remain eigenstates whose energies vary linearly with field strength. Once the field reaches `2 * exchange_rate`, the spin-aligned triplet drops below the singlet, switching the ground state from maximally entangled zero net spin to an unentangled polarized pair.

In bra-ket notation, these levels are the eigenvalues of $H = \tfrac{J}{2}\left(I + \boldsymbol\sigma_1\cdot\boldsymbol\sigma_2\right) - \tfrac{B}{2}\left(\sigma_{1z} + \sigma_{2z}\right)$ across the antiferromagnetic dimer level crossing.

In [ ]:
def energy(state: Spinor, exchange_rate: np.ndarray, field: np.ndarray) -> Scalar:
    """Shifted exchange coupling and a uniform xy-plane magnetic field give the expected state energy (`[...] Scalar+`)."""
    first_plane, second_plane = spin_planes(state)                              # [...] FirstPlanes+, SecondPlanes+
    return exchange_rate * expectation(0.5 * (1 - coupling), state) - 0.5 * field * field_plane.dot(first_plane + second_plane)   # [...] Scalar+


# Four states of definite spin in the xy planes: singlet, zero-spin triplet, both up, and both down.
basis_states = stack([
    singlet,
    np.sqrt(2) * triplet_part * up_down,
    correlator,
    mv.zx * mv.ZX * correlator,
])                                                                              # [4] Spinor+
basis_first, basis_second = spin_planes(basis_states)                           # [4] FirstPlanes+, SecondPlanes+
total_spins = field_plane.dot(basis_first + basis_second)                       # [4] Scalar+
exchange_rate = np.asarray(1.0)
n_fields = 201
fields = np.linspace(0.0, 4.0, n_fields)
energies = energy(basis_states, exchange_rate, fields[:, None])                 # [n_fields, 4] Scalar+
save_figure(render.draw_levels(fields, energies, total_spins), "two_spins_levels");

On the left, the purple singlet and grey zero-spin triplet stay flat while the red spin-aligned and blue spin-opposed triplets split linearly, with the faint grey band tracing the ground-state energy through the crossing at `field == 2`. On the right, the ground-state spin jumps from 0 to 2 at that crossing.

## Singlet-triplet qubit

A magnetic field difference `difference * difference_plane` between two neighbouring sites turns `xy` against `XY`, leaving the polarized triplets untouched while coupling the singlet to the zero-spin triplet, as realized in semiconductor double quantum dots. On this two-state subspace, applying `time_generator` twice multiplies every state by `-(exchange_rate**2 + difference**2)`, so time evolution closes into `cos(rate * time) * state + sin(rate * time) / rate * time_generator(...)` and precesses on a two-spin qubit sphere.

In bra-ket notation ([Hanson et al., 2007](https://arxiv.org/abs/cond-mat/0610433)), $-iH$ with $H = \tfrac{J}{2}\left(I + \boldsymbol\sigma_1\cdot\boldsymbol\sigma_2\right) + \tfrac{\Delta}{2}\left(\sigma_{1z} - \sigma_{2z}\right)$ reduces on $\{|S\rangle, |T_0\rangle\}$ to an effective $2 \times 2$ Pauli Hamiltonian $J\,\tau_z + \Delta\,\tau_x$ with singlet return probability $1 - \tfrac{\Delta^2}{J^2 + \Delta^2}\sin^2\!\left(\sqrt{J^2 + \Delta^2}\,t\right)$.

In [ ]:
def time_generator(state: Spinor, exchange_rate: np.ndarray, field: Bivector) -> Spinor:
    """Shifted exchange about `reference_spin` and spatial bivector `field` drive the infinitesimal time rate (`[...] Spinor+`) of `state`."""
    return -exchange_rate * (0.5 * (1 - coupling) * state) * reference_spin - field * state   # [...] Spinor+


def qubit_state(state: Spinor, exchange_rate: np.ndarray, difference: np.ndarray, times: np.ndarray) -> Spinor:
    """Harmonic precession under `exchange_rate` and field `difference` carries a zero-net-spin `state` to `times` (`[...] Spinor+`)."""
    rate = np.sqrt(exchange_rate**2 + difference**2)
    step_generator = time_generator(state, exchange_rate, difference * difference_plane)   # [...] Spinor+
    return np.cos(rate * times) * state + np.sin(rate * times) / rate * step_generator     # [...] Spinor+


difference = np.asarray(1.0)
differential_field = difference * difference_plane                              # [] Bivector+
exchange_rates = np.array([0.0, 1.0, 3.0])
n_times = 241
times = np.linspace(0.0, np.pi, n_times)
qubit_states = qubit_state(singlet, exchange_rates[:, None], difference, times) # [3, n_times] Spinor+
singlet_return = expectation(singlet_part, qubit_states)                        # [3, n_times] Scalar+
subspace_pair = basis_states[:2]                                                # [2] Spinor+
repeated_generator = time_generator(
    time_generator(subspace_pair, exchange_rates[1], differential_field),
    exchange_rates[1],
    differential_field,
)                                                                               # [2] Spinor+

In [ ]:
def qubit_coordinates(state: Spinor) -> tuple[Scalar, Scalar, Scalar]:
    """Spin imbalance, relative phase, and singlet-triplet balance place `state` on the two-spin qubit sphere (`[...] Scalar+` each)."""
    first_plane, second_plane = spin_planes(state)                              # [...] FirstPlanes+, SecondPlanes+
    imbalance = 0.5 * (mv.xy.dot(first_plane) - mv.XY.dot(second_plane))        # [...] Scalar+
    phase_coordinate = expectation(-0.25 * (1 - coupling) * (mv.xy - mv.XY), state)   # [...] Scalar+
    singlet_balance = expectation(0.5 * (coupling - 1), state)                  # [...] Scalar+
    return imbalance, phase_coordinate, singlet_balance


imbalance, phase_coordinate, singlet_balance = qubit_coordinates(qubit_states)  # [3, n_times] Scalar+, Scalar+, Scalar+
qubit_first, qubit_second = spin_planes(qubit_states)                           # [3, n_times] FirstPlanes+, SecondPlanes+
save_figure(render.draw_qubit(imbalance, phase_coordinate, singlet_balance, times, singlet_return, exchange_rates), "two_spins_qubit");

Starting from the black dot at the singlet pole, pure differential driving in purple orbits through the zero-spin triplet at the south pole while staying maximally entangled, equal exchange and field difference in red tilts the precession axis to pass through the unentangled `up-down` and `down-up` states on the equator, and strong exchange in blue confines the trajectory near the singlet pole. The right panel plots the corresponding singlet return probability oscillations.

In [ ]:
# checks
# Correlator, reference spin, spacelike commutation with t and T, and exchange projectors.
assert ((correlator * correlator - correlator).scalar_norm_squared() < 1e-12).all()
assert ((correlator * mv.xy - correlator * mv.XY).scalar_norm_squared() < 1e-12).all()
assert ((reference_spin * reference_spin + correlator).scalar_norm_squared() < 1e-12).all()
assert ((state_norm - 1).abs() < 1e-12).all()
assert ((mv.t * states - states * mv.t).scalar_norm_squared().abs() < 1e-12).all()
assert ((mv.T * states - states * mv.T).scalar_norm_squared().abs() < 1e-12).all()
assert ((coupling * coupling - 3 - 2 * coupling).scalar_norm_squared() < 1e-12).all()
assert ((singlet_part * singlet_part - singlet_part).scalar_norm_squared() < 1e-12).all()
assert ((triplet_part * triplet_part - triplet_part).scalar_norm_squared() < 1e-12).all()
assert ((singlet_part + triplet_part - 1).scalar_norm_squared() < 1e-12).all()

# Exchange normalization, midpoint maximal entanglement, spin swap, and Bell square identity.
assert ((2 * states.scalar_norm_squared() - 1).abs() < 1e-12).all()
assert ((start_bell - 2).abs() < 1e-12).all()
assert (first_planes[midpoint].scalar_norm_squared() < 1e-12).all()
assert ((bells[midpoint] - 2 * np.sqrt(2)).abs() < 1e-12).all()
assert ((first_planes[-1] + mv.xy).scalar_norm_squared() < 1e-12).all()
assert ((second_planes[-1] - mv.XY).scalar_norm_squared() < 1e-12).all()
assert ((coupling * singlet - 3 * singlet).scalar_norm_squared() < 1e-12).all()
for second_unit, first_unit in ((mv.YZ, mv.yz), (mv.ZX, mv.zx), (mv.XY, mv.xy)):
    assert ((singlet_joint(second_unit) + first_unit).scalar_norm_squared() < 1e-12).all()
assert ((element_square - commutator_bound).scalar_norm_squared() < 1e-12).all()
assert ((singlet_bell - 2 * np.sqrt(2)).abs() < 1e-12).all()

# Steering adjoint formula, local singular-value invariance, field crossing, and qubit precession.
assert ((steered_planes - adjoint_planes).scalar_norm_squared() < 1e-12).all()
assert ((probability - 0.5 * (1 + planes.dot(first_planes[:, None, None]))).abs() < 1e-12).all()
assert ((turned_joints - sandwich_joints).svdvals().abs() < 1e-9).all()
assert ((turned_singular_values - singular_values).abs() < 1e-9).all()
assert ((first_planes.scalar_norm_squared() + concurrence**2 - 1).abs() < 1e-9).all()
assert ((total_spins - np.array([0.0, 0.0, 2.0, -2.0])).abs() < 1e-12).all()
assert ((energies[n_fields // 2] - np.array([-1.0, 1.0, -1.0, 3.0])).abs() < 1e-12).all()
equal_rate = np.hypot(exchange_rates[1], difference)
assert ((repeated_generator + equal_rate**2 * subspace_pair).scalar_norm_squared() < 1e-12).all()
assert ((imbalance * imbalance + phase_coordinate * phase_coordinate + singlet_balance * singlet_balance - 1).abs() < 1e-12).all()
assert (qubit_first[0].scalar_norm_squared() < 1e-12).all()
generalized_rates = np.hypot(exchange_rates[:, None], difference)
expected_return = 1 - (difference / generalized_rates) ** 2 * np.sin(generalized_rates * times) ** 2
assert ((singlet_return - expected_return).abs() < 1e-12).all()

# Mirror equivariance across both spacetime copies without pseudoscalars.
mirrors = (
    (mv.x * mv.X, mv.yz * mv.YZ),
    (mv.z * mv.Z, mv.xy * mv.XY),
    ((half * (mv.x + mv.z)) * (half * (mv.X + mv.Z)), (half * (mv.yz + mv.xy)) * (half * (mv.YZ + mv.XY))),
)
for reflection, half_turn in mirrors:
    mirrored_first, mirrored_second = spin_planes(half_turn * states)           # [n_angles] FirstPlanes+, SecondPlanes+
    assert ((half_turn * states * correlator - half_turn * states).scalar_norm_squared() < 1e-12).all()
    assert ((mirrored_first - (reflection >> first_planes)).scalar_norm_squared() < 1e-12).all()
    assert ((mirrored_second - (reflection >> second_planes)).scalar_norm_squared() < 1e-12).all()
    assert ((exchange_state(half_turn * up_down, angles) - half_turn * states).scalar_norm_squared() < 1e-12).all()
    mirrored_step = time_generator(half_turn * qubit_states, exchange_rates[:, None], reflection >> differential_field)   # [3, n_times] Spinor+
    expected_step = half_turn * time_generator(qubit_states, exchange_rates[:, None], differential_field)                 # [3, n_times] Spinor+
    assert ((mirrored_step - expected_step).scalar_norm_squared() < 1e-12).all()